# Fecha de quiebra de cada empresa

Este notebook busca, para cada empresa de `data/sub.csv`, la fecha en que pidió la quiebra dentro del período de estudio.

**La fuente.** Desde agosto de 2004, una empresa que pide la quiebra (Chapter 7 o Chapter 11) presenta un Form 8-K con el ítem 1.03 ("Bankruptcy or Receivership") dentro de los 4 días hábiles. Los 8-K no están en los zips: la base de la SEC solo trae documentos con estados contables en XBRL. Por eso la consulta usa la API de la SEC (`data.sec.gov/submissions`).

La etiqueta usa solo el 8-K. Es la misma fuente de la base comercial de quiebras de Audit Analytics.

Pasos:

1. Elegir las empresas: todas las de `sub.csv`.
2. Bajar la lista de documentos de cada empresa.
3. Quedarse con los 8-K con el ítem 1.03 de una empresa.
4. Repetirlo para todas las empresas y guardar los 8-K en `data/bankruptcy_filings.csv`.
5. Elegir la fecha de quiebra dentro del período de estudio.

In [1]:
import sys
import time
from pathlib import Path

import pandas as pd
import requests


if "google.colab" in sys.modules:
    from google.colab import drive  # type: ignore

    drive.mount("/content/drive")

    CANDIDATOS = [
        Path("/content/drive/MyDrive/Colab Notebooks/Diplodatos MAI Grupo 6"),
        Path("/content/drive/MyDrive/Diplodatos MAI Grupo 6"),
    ]
    WORKING_DIR = next((p for p in CANDIDATOS if p.exists()), None)
    if WORKING_DIR is None:
        raise FileNotFoundError(
            "No encuentro la carpeta del grupo. ¿Agregaste el acceso directo "
            "en 'Mi unidad' desde 'Compartido conmigo'?"
        )
else:
    WORKING_DIR = Path.cwd()

## Paso 1: empresas

Se consultan **todas** las empresas, no solo las que dejan de presentar. Una empresa en Chapter 11 (reorganización) sigue operando y presenta sus 10-Q. Gulfport y Chord/Oasis pidieron la quiebra en 2020 y siguieron presentando sus estados contables.

El sector y el nombre salen del documento más reciente de cada empresa, como en `build_panel.ipynb`.

In [2]:
sub = pd.read_csv(f"{WORKING_DIR}/data/sub.csv", dtype=str)

# one row per company: name and sector of its most recent document (sorted by `filed`)
companies = sub.sort_values("filed").groupby("cik")[["name", "sector"]].last()
companies["sector"].value_counts()
companies.head(5)

,name,sector
cik,,
1000229,CORE LABORATORIES N V,energy
1001614,"RILEY EXPLORATION PERMIAN, INC.",energy
1004724,"REGIONAL HEALTH PROPERTIES, INC",health
10048,BARNWELL INDUSTRIES INC,energy
1006655,EVOLUTION PETROLEUM CORP,energy


## Paso 2: documentos de una empresa

`data.sec.gov/submissions/CIK##########.json` devuelve los documentos de una empresa, con el CIK completado con ceros hasta 10 dígitos. Cada 8-K tiene el campo `items` con los ítems que la empresa declara, por ejemplo `"1.03,9.01"`.

- `filings.recent` trae los últimos 1.000 documentos, aproximadamente.
- Los más viejos están en archivos extra, listados en `filings.files`. La función los lee también.
- La SEC acepta hasta 10 consultas por segundo y rechaza la consulta (error 403) si el `User-Agent` no tiene un email, igual que en `download.ipynb`.

In [3]:
EMAIL = "ejemplo@ejemplo.com"  # no se verifica el correo, pero es necesario (igual que en download.ipynb)
HEADERS = {"User-Agent": f"Diplodatos UNC {EMAIL}"}
BASE = "https://data.sec.gov/submissions/"


def get(path: str) -> dict:
    """One request to the submissions API."""
    time.sleep(0.1)  # SEC limit: 10 requests per second
    print(f"GET {BASE + path}")
    response = requests.get(BASE + path, headers=HEADERS, timeout=30)
    response.raise_for_status()
    return response.json()


def filings(cik: str) -> pd.DataFrame:
    """All filings of a company: `recent` plus the older extra files."""
    cik_10 = str(int(cik)).zfill(10)  # the API uses the CIK with 10 digits
    company = get(f"CIK{cik_10}.json")

    # latest ~1,000 filings
    pages = [pd.DataFrame(company["filings"]["recent"])]

    # older filings, one extra file each
    for extra in company["filings"]["files"]:
        pages.append(pd.DataFrame(get(extra["name"])))

    return pd.concat(pages, ignore_index=True)

## Paso 3: 8-K con el ítem 1.03 de una empresa

`bankruptcy_filings` devuelve las fechas de **todos** los 8-K con el ítem 1.03, de la más vieja a la más nueva. Una empresa puede tener varios:

- **Un mismo caso de quiebra tiene más de un 8-K.** El ítem 1.03 también informa la aprobación del plan de reorganización.
- **Una empresa puede quebrar más de una vez.** Por ejemplo, en 2009 y en 2020.

El paso 5 elige cuál de las fechas es la quiebra del período.

Ejemplo: Gulfport (CIK 874499) tiene dos 8-K con el ítem 1.03:

| `filingDate` | Qué es |
|---|---|
| 2020-11-16 | Pedido de quiebra |
| 2021-04-29 | Aprobación del plan de reorganización |

La celda termina con un control sobre Gulfport.

In [4]:
def bankruptcy_filings(cik: str) -> list[str]:
    """Filing dates of all 8-Ks with item 1.03 (bankruptcy), oldest first."""
    f = filings(cik)
    is_8k = f["form"].str.startswith("8-K")  # includes amendments (8-K/A)
    has_item = f["items"].str.contains("1.03", regex=False)
    return sorted(f.loc[is_8k & has_item, "filingDate"])


assert bankruptcy_filings("874499") == ["2020-11-16", "2021-04-29"]  # Gulfport

GET https://data.sec.gov/submissions/CIK0000874499.json


GET https://data.sec.gov/submissions/CIK0000874499-submissions-001.json


## Paso 4: todas las empresas

La consulta tarda unos minutos: una consulta por empresa, más los archivos extra. El resultado se guarda en `data/bankruptcy_filings.csv`, con una fila por 8-K con el ítem 1.03 (`cik`, `filing_date`). Si el archivo ya existe, el notebook lo lee y no consulta otra vez a la SEC.

El archivo guarda los 8-K de toda la historia, sin filtrar por período. Así, si el período de estudio cambia, alcanza con ejecutar de nuevo el paso 5.

El paso 5 guarda el resultado en `data/bankruptcy.csv` (`cik`, `name`, `sector`, `bankruptcy_date`), que lee `build_panel.ipynb`.

In [5]:
FILE = Path(f"{WORKING_DIR}/data/bankruptcy_filings.csv")

# If the file already exists, do not query the SEC again.
if FILE.exists():
    found = pd.read_csv(FILE, dtype=str)
else:
    rows = []
    for cik in companies.index:
        for date in bankruptcy_filings(cik):
            rows.append({"cik": cik, "filing_date": date})
    found = pd.DataFrame(rows, columns=["cik", "filing_date"])
    found.to_csv(FILE, index=False)

found

,cik,filing_date
0,1022646,2016-05-03
1,1022646,2017-03-16
2,1022646,2020-05-15
3,1022646,2020-05-21
4,1022646,2020-08-25
...,...,...
194,945764,2020-07-31
195,945764,2020-09-04
196,949039,2020-04-27
197,949039,2020-04-28


## Paso 5: fecha de quiebra dentro del período

El período del panel es 2018Q1–2024Q4 (`docs/rumbo-mai.pdf`). `END` llega hasta el 31/12/2025 porque el objetivo es predecir la quiebra en los próximos 2 trimestres o 1 año: una empresa del panel en 2024Q4 necesita saber si quiebra en 2025. Una quiebra posterior no interesa, por ejemplo una de 2026.

**Regla.** Un 8-K abre un caso de quiebra nuevo si el 8-K con el ítem 1.03 anterior tiene más de 2 años. Si no, es parte del mismo caso, por ejemplo la aprobación del plan. La fecha de quiebra es el primer caso nuevo que empieza dentro del período.

Con esta regla:

- Gulfport da 2020-11-16. El 8-K de 2021-04-29 es del mismo caso.
- Una empresa que quebró en 2017 y tiene la aprobación del plan en 2018 no tiene quiebra en el período. La quiebra empezó antes.
- Una empresa que quebró en 2009 y en 2020 da 2020-11-16.

La celda controla la regla con estos casos antes de aplicarla.

In [6]:
START, END = "2018-01-01", "2025-12-31"  # panel 2018Q1-2024Q4 plus 1 year of prediction horizon
# ponytail: fixed 2-year gap; a case with plan approval >2 years after the petition counts twice
NEW_CASE_GAP = pd.Timedelta(days=730)


def bankruptcy_date(dates: list[str]) -> str | None:
    """Start of the first bankruptcy case inside the study period, or None."""
    previous = None
    for date in sorted(dates):
        is_new_case = previous is None or pd.Timestamp(date) - pd.Timestamp(previous) > NEW_CASE_GAP
        previous = date
        if is_new_case and START <= date <= END:
            return date
    return None


assert bankruptcy_date(["2020-11-16", "2021-04-29"]) == "2020-11-16"  # Gulfport
assert bankruptcy_date(["2017-06-01", "2018-03-01"]) is None  # case started before the period
assert bankruptcy_date(["2009-06-01", "2009-12-01", "2020-11-16"]) == "2020-11-16"  # older case skipped
assert bankruptcy_date(["2026-02-01"]) is None  # after the period
assert bankruptcy_date([]) is None

dates_by_cik = found.groupby("cik")["filing_date"].apply(list)
in_period = dates_by_cik.apply(bankruptcy_date).dropna().rename("bankruptcy_date")
bankruptcy = companies.join(in_period)  # NaN: no bankruptcy in the period

bankruptcy.to_csv(f"{WORKING_DIR}/data/bankruptcy.csv")
print(bankruptcy.groupby("sector")["bankruptcy_date"].count())
bankruptcy.dropna(subset=["bankruptcy_date"]).sort_values("bankruptcy_date")

sector
energy    52
health    18
Name: bankruptcy_date, dtype: int64


,name,sector,bankruptcy_date
cik,,,
316300,EXCO RESOURCES INC,energy,2018-01-17
1361937,HARVEST OIL & GAS CORP.,energy,2018-04-02
1402281,ERIN ENERGY CORP.,energy,2018-04-26
1397516,REX ENERGY CORP,energy,2018-05-18
741815,HOOPER HOLMES INC,health,2018-08-28
...,...,...,...
1651944,"DERMTECH, INC.",health,2024-06-18
1431959,TORCHLIGHT ENERGY RESOURCES INC,energy,2024-08-09
1813914,"CAREMAX, INC.",health,2024-11-18


## Limitaciones

- **Empresas que dejan de presentar antes de la quiebra.** No presentan el 8-K, así que el notebook no las detecta.
- **Quiebras de filiales.** Un 8-K con el ítem 1.03 puede informar la quiebra de una filial y no la de la empresa. El notebook la cuenta como quiebra de la empresa.
- **Regla de 2 años.** Si un mismo caso tiene dos 8-K separados por más de 2 años, cuenta como dos quiebras. Si una empresa quiebra dos veces en menos de 2 años, cuenta como una.
- **Chapter 7 o Chapter 11.** El ítem 1.03 no los distingue. El Chapter 7 es la liquidación y el Chapter 11 la reorganización. Separarlos obliga a leer el texto del 8-K.
- **Compras y otras salidas.** Este notebook solo busca la quiebra. Una compra se detecta con el ítem 2.01 junto con la salida de la empresa (Form 15 o Form 25 y ninguna presentación posterior). PRECAUCIÓN: el ítem 2.01 también aparece cuando la empresa es la compradora.